# 06-03 多 Agent 对话编排

AutoGen 的强项：多个 Agent 之间的**有组织对话**。

**本节目标**：GroupChat、Speaker Selection、Termination Conditions

---

In [ ]:
import os, sys
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

try:
    from autogen import ConversableAgent, GroupChat, GroupChatManager
    HAS_AG = True
except ImportError:
    HAS_AG = False
    print("AutoGen 未安装")

## 1. GroupChat（多 Agent 群聊）

In [ ]:
llm_config = {"config_list": [{"model": "gpt-4o-mini", "api_key": os.environ.get("OPENAI_API_KEY", "")}]}

if HAS_AG:
    # 创建专家团队
    data_analyst = ConversableAgent(
        name="数据分析师",
        system_message="你是广告数据分析师，专注CTR/CVR/ROI分析。简洁回答（2-3句）。",
        llm_config=llm_config,
    )
    
    creative_writer = ConversableAgent(
        name="文案创作师",
        system_message="你是广告文案师，专注B站风格的创意文案。给出具体文案（标题+正文）。",
        llm_config=llm_config,
    )
    
    compliance_officer = ConversableAgent(
        name="合规审核员",
        system_message="你是广告合规审核员，检查极限词和虚假宣传。只说问题或'通过'。",
        llm_config=llm_config,
    )
    
    # 创建 GroupChat
    group_chat = GroupChat(
        agents=[data_analyst, creative_writer, compliance_officer],
        messages=[],
        max_round=6,
        speaker_selection_method="auto",  # LLM 自动选择下一个发言者
    )
    
    manager = GroupChatManager(
        groupchat=group_chat,
        llm_config=llm_config,
    )
    
    print("GroupChat 创建成功")
    print(f"  Agent: {[a.name for a in group_chat.agents]}")
    print(f"  最大轮数: {group_chat.max_round}")
else:
    print("""
GroupChat 架构:
  agents = [analyst, writer, reviewer]  # 多个专家 Agent
  
  group_chat = GroupChat(
      agents=agents,
      max_round=6,
      speaker_selection_method="auto",  # 或 "round_robin", "random"
  )
  
  manager = GroupChatManager(groupchat=group_chat)
  agent_a.initiate_chat(manager, message="...")

Speaker 选择策略:
  auto:        LLM 根据对话上下文选择（最智能）
  round_robin: 轮流发言
  random:      随机选择
  manual:      人工选择
    """)

In [ ]:
# 运行 GroupChat
if HAS_AG and os.environ.get("OPENAI_API_KEY"):
    result = data_analyst.initiate_chat(
        manager,
        message="游戏广告CTR 1.2%，需要创作新文案并审核。广告预算日均500元。",
    )
    print(f"\n对话轮数: {len(result.chat_history)}")
else:
    print("""
模拟 GroupChat 流程:
  [数据分析师] CTR 1.2% 低于均值 2.1%，建议优化创意素材，当前 CPC 偏高
  [文案创作师] 新方案 - 标题：沉浸游戏体验 限时畅玩  正文：精选皮肤限时折扣...
  [合规审核员] 通过，无极限词和虚假宣传
  [数据分析师] 建议 A/B 测试新旧创意，预期 CTR 可提升至 2-3%
    """)

## 2. 终止条件

In [ ]:
print("""
AutoGen 终止条件设计:

1. max_turns / max_round: 最大对话轮数
   result = a.initiate_chat(b, max_turns=5)

2. is_termination_msg: 根据消息内容终止
   agent = ConversableAgent(
       is_termination_msg=lambda msg: "TERMINATE" in msg["content"]
   )

3. max_consecutive_auto_reply: 连续自动回复上限
   agent = ConversableAgent(max_consecutive_auto_reply=3)

4. 自定义终止函数:
   def should_stop(msg):
       return "审核通过" in msg.get("content", "")
""")

## AutoGen 核心概念

| 概念 | 说明 |
|------|------|
| ConversableAgent | 基础 Agent 类，可发送/接收消息 |
| GroupChat | 多 Agent 群聊容器 |
| GroupChatManager | 管理群聊流程、选择发言者 |
| register_function | 注册工具给 Agent |
| initiate_chat | 发起对话 |

## 面试速记

| 问题 | 要点 |
|------|------|
| AutoGen 的核心设计 | 万物皆对话，Agent 通过消息传递协作 |
| GroupChat speaker 选择 | auto(LLM选), round_robin(轮流), random, manual |
| AutoGen vs LangGraph | AutoGen 适合对话场景；LangGraph 适合工作流编排 |

**Module 06 完成！** 下一步：`../07-agent-platforms/`